In [1]:
import numpy as np
import cv2

In [9]:
try:
    cap.release()
except NameError:
    pass
cap = cv2.VideoCapture(0,cv2.CAP_DSHOW)
assert cap.isOpened()


In [8]:
cv2.imshow("Camera Feed", cap.read()[1])

if cv2.waitKey(0) == ord('q'):
    cv2.destroyAllWindows()


In [6]:
# 暂时没有用到，可以用来把靶环给滤掉
def hls_filter(frame:np.ndarray,**kwargs) -> np.ndarray:
    """
    HLS颜色空间滤波
    :param frame: 输入图像
    :param kwargs: 可选参数，包含以下键值对：
        - h: H通道的阈值范围，格式为 (h_min, h_max)，默认 (0, 180)
        - l: L通道的阈值范围，格式为 (l_min, l_max)，默认 (0, 255)
        - s: S通道的阈值范围，格式为 (s_min, s_max)，默认 (0, 255)
    :return: 滤波后的二值图像
    """
    hls_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HLS)
    h, l, s = cv2.split(hls_frame)
    h_min, h_max = kwargs.get('h', (0, 180))
    l_min, l_max = kwargs.get('l', (0, 255))
    s_min, s_max = kwargs.get('s', (0, 255))

    if h_min < h_max:
        upper = np.array([h_max, l_max, s_max], dtype=np.uint8)
        lower = np.array([h_min, l_min, s_min], dtype=np.uint8)
        mask = cv2.inRange(hls_frame, lower, upper)
    else:
        upper1 = np.array([180, l_max, s_max], dtype=np.uint8)
        lower1 = np.array([h_min, l_min, s_min], dtype=np.uint8)
        upper2 = np.array([h_max, l_max, s_max], dtype=np.uint8)
        lower2 = np.array([0, l_min, s_min], dtype=np.uint8)
        mask1 = cv2.inRange(hls_frame, lower1, upper1)
        mask2 = cv2.inRange(hls_frame, lower2, upper2)
        mask = cv2.bitwise_or(mask1, mask2)

    return mask

### 第一版 自适应阈值+形态学+膨胀+检测（容易丢）

In [7]:
cap.set(cv2.CAP_PROP_AUTO_EXPOSURE, 0.25)  # Set auto exposure to manual mode
cap.set(cv2.CAP_PROP_EXPOSURE, -8)  # Set exposure value (adjust as needed)
kernel = np.ones((3,3), np.uint8)
while True:

    # 画出视野中点
    # height, width, _ = video.read()[1].shape
    # img = cv2.circle(video.read()[1], (center_x, center_y), 5, (0, 255, 0), -1)
    # center_x, center_y = width // 2, height // 2
    
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break
    frame = cv2.resize(frame, (320,240))
    cv2.imshow("Camera Feed", frame)
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # hls_mask = hls_filter(frame, s=(0, 50))
    # cv2.imshow("HLS Filtered", hls_mask)
    img = cv2.GaussianBlur(img, (5, 5), 0)
    # ostubimg = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    bimg = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11,7)
    bimg = cv2.morphologyEx(bimg, cv2.MORPH_OPEN, kernel,iterations=1)
    # bimg = cv2.bitwise_or(bimg, ostubimg)
    bimg = cv2.dilate(bimg, kernel, iterations=1)
    
    # 矩形检测
    contours, _ = cv2.findContours(bimg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # 在原图上绘制检测到的矩形
    result_frame = frame.copy()
    
    for contour in contours:
        # 计算轮廓面积，过滤小的噪声
        area = cv2.contourArea(contour)
        if area > 500 :  # 可以调整这个阈值
            # 计算轮廓的近似多边形
            epsilon = 0.02 * cv2.arcLength(contour, True)
            approx = cv2.approxPolyDP(contour, epsilon, True)
            
            # 如果近似多边形有4个顶点，认为是矩形
            if len(approx) == 4:
                # 绘制矩形轮廓
                cv2.drawContours(result_frame, [approx], -1, (0, 255, 0), 2)
                
                # 计算外接矩形
                x, y, w, h = cv2.boundingRect(contour)
                cv2.rectangle(result_frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
                
                # 在矩形上方显示面积信息
                cv2.putText(result_frame, f'Area: {int(area)}', (x, y-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
    
    cv2.imshow("bimg", bimg)
    cv2.imshow("Rectangle Detection", result_frame)
    

    if cv2.waitKey(1) == ord('q'):
        cv2.destroyAllWindows()
        break

### 第二版：不用形态学，添加凸多边形检测，非常优秀👍完全不会误检测和丢

In [14]:

cap.set(cv2.CAP_PROP_AUTO_EXPOSURE, 1)  # Set auto exposure to manual mode
cap.set(cv2.CAP_PROP_EXPOSURE, -8)  # Set exposure value (adjust as needed)
kernel = np.ones((3,3), np.uint8)
while True:

    # 画出视野中点
    # height, width, _ = video.read()[1].shape
    # img = cv2.circle(video.read()[1], (center_x, center_y), 5, (0, 255, 0), -1)
    # center_x, center_y = width // 2, height // 2
    
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break
    frame = cv2.resize(frame, (320,240))
    cv2.imshow("Camera Feed", frame)
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # hls_mask = hls_filter(frame, s=(0, 50))
    # cv2.imshow("HLS Filtered", hls_mask)
    img = cv2.GaussianBlur(img, (5, 5), 0)
    bimg = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11,7)
    
    # 矩形检测
    contours, _ = cv2.findContours(bimg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # 在原图上绘制检测到的矩形
    result_frame = frame.copy()
    
    for contour in contours:
        # 计算轮廓面积，过滤小的噪声
        area = cv2.contourArea(contour)
        if area > 500 :  # 可以调整这个阈值
            # 计算轮廓的近似多边形
            epsilon = 0.02 * cv2.arcLength(contour, True)
            approx = cv2.approxPolyDP(contour, epsilon, True)
            
            # 如果近似多边形有4个顶点，认为是矩形
            if len(approx) == 4:
                # 检查是否为凸多边形
                is_convex = cv2.isContourConvex(approx)
                
                if is_convex:  # 只保留凸多边形
                    # 计算凸包面积与轮廓面积的比值来进一步验证凸性
                    hull = cv2.convexHull(contour)
                    hull_area = cv2.contourArea(hull)
                    convexity_ratio = area / hull_area if hull_area > 0 else 0
                    
                    # 凸性比值应该接近1（凸多边形的面积与其凸包面积相等）
                    if convexity_ratio > 0.92:  # 调整阈值以过滤轻微的凹陷
                        # 绘制矩形轮廓
                        cv2.drawContours(result_frame, [approx], -1, (0, 255, 0), 2)
                        
                        # 计算外接矩形
                        x, y, w, h = cv2.boundingRect(contour)
                        cv2.rectangle(result_frame, (x, y), (x + w, y + h), (255, 0, 0), 2)
                        
                        # 在矩形上方显示面积信息和凸性信息
                        cv2.putText(result_frame, f'Area: {int(area)}', (x, y-25), 
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)
                        cv2.putText(result_frame, f'Convex: {convexity_ratio:.2f}', (x, y-10), 
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
                    else:
                        # 标记被过滤的非凸形状（可选，用于调试）
                        cv2.drawContours(result_frame, [approx], -1, (0, 0, 255), 1)
                        cv2.putText(result_frame, 'Non-convex', (approx[0][0][0], approx[0][0][1]), 
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 0, 255), 1)
                else:
                    # 标记被过滤的非凸形状（可选，用于调试）
                    cv2.drawContours(result_frame, [approx], -1, (128, 128, 128), 1)
                    cv2.putText(result_frame, 'Concave', (approx[0][0][0], approx[0][0][1]), 
                               cv2.FONT_HERSHEY_SIMPLEX, 0.3, (128, 128, 128), 1)
    
    cv2.imshow("bimg", bimg)
    cv2.imshow("Rectangle Detection", result_frame)
    

    if cv2.waitKey(1) == ord('q'):
        cv2.destroyAllWindows()
        break

分辨率不要开高，不然自适应阈值容易把黑色干掉，要大大增加blocksize

### 第三版：添加了矩形中心检测、距离估计和云台角度计算功能

In [15]:
# 云台配置参数
class GimbalConfig:
    def __init__(self):
        # 摄像头参数
        self.image_width = 320
        self.image_height = 240
        self.image_center_x = self.image_width // 2
        self.image_center_y = self.image_height // 2
        
        # 摄像头视野角度 (根据你的摄像头规格调整)
        self.camera_fov_horizontal = 90  # 水平视野角度（度）
        self.camera_fov_vertical = 64    # 垂直视野角度（度）
        
        # 已知矩形的实际尺寸 (单位：毫米，根据实际目标调整)
        self.real_rect_width = 315   # 实际矩形宽度
        self.real_rect_height = 225   # 实际矩形高度
        
        # 角度转换系数 (像素/度)
        self.pixels_per_degree_x = self.image_width / self.camera_fov_horizontal
        self.pixels_per_degree_y = self.image_height / self.camera_fov_vertical

# 创建配置实例
config = GimbalConfig()

def calculate_distance_from_rect(rect_width_pixels, rect_height_pixels, config):
    """
    根据矩形在图像中的像素尺寸估计距离
    
    :param rect_width_pixels: 矩形在图像中的像素宽度
    :param rect_height_pixels: 矩形在图像中的像素高度
    :param config: 云台配置对象
    :return: 估计的距离 (毫米)
    """
    # 使用相似三角形原理计算距离
    # distance = (real_size * focal_length) / pixel_size
    # 这里用视野角度来近似计算
    
    # 计算水平和垂直方向的距离估计
    distance_from_width = (config.real_rect_width * config.image_width) / (rect_width_pixels * 2 * np.tan(np.radians(config.camera_fov_horizontal / 2)))
    distance_from_height = (config.real_rect_height * config.image_height) / (rect_height_pixels * 2 * np.tan(np.radians(config.camera_fov_vertical / 2)))
    
    # 取平均值作为最终距离估计
    estimated_distance = (distance_from_width + distance_from_height) / 2
    
    return estimated_distance

def calculate_gimbal_angles(rect_center_x, rect_center_y, estimated_distance, config):
    """
    计算云台需要旋转的角度来瞄准矩形中心
    
    :param rect_center_x: 矩形中心x坐标
    :param rect_center_y: 矩形中心y坐标
    :param estimated_distance: 估计的目标距离
    :param config: 云台配置对象
    :return: (yaw_angle, pitch_angle) 云台需要旋转的角度（度）
    """
    # 计算像素偏差
    pixel_offset_x = rect_center_x - config.image_center_x
    pixel_offset_y = config.image_center_y - rect_center_y  # 注意y轴方向
    
    # 转换为角度偏差
    yaw_angle = pixel_offset_x / config.pixels_per_degree_x
    pitch_angle = pixel_offset_y / config.pixels_per_degree_y
    
    return yaw_angle, pitch_angle

def get_rectangle_center(contour):
    """
    获取矩形的中心点
    
    :param contour: 轮廓点
    :return: (center_x, center_y) 矩形中心坐标
    """
    # 方法1：使用轮廓的矩形边界框
    x, y, w, h = cv2.boundingRect(contour)
    center_x = x + w // 2
    center_y = y + h // 2
    
    # 方法2：使用轮廓的质心（更精确）
    M = cv2.moments(contour)
    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        return cx, cy
    else:
        return center_x, center_y

print("云台配置和计算函数已加载完成！")

云台配置和计算函数已加载完成！


In [16]:
# 云台控制功能示例
def send_gimbal_command(yaw_angle, pitch_angle):
    """
    发送云台控制命令
    这是一个示例函数，你需要根据你的云台通信协议来实现
    
    :param yaw_angle: 水平旋转角度
    :param pitch_angle: 垂直旋转角度
    """
    # 示例：打印命令（实际使用时替换为串口通信或其他通信方式）
    print(f"Gimbal Command - Yaw: {yaw_angle:.2f}°, Pitch: {pitch_angle:.2f}°")
    
    # 示例：如果使用串口通信
    # import serial
    # ser = serial.Serial('COM3', 9600)  # 替换为你的串口
    # command = f"YAW:{yaw_angle:.2f},PITCH:{pitch_angle:.2f}\n"
    # ser.write(command.encode())
    
    # 示例：如果使用socket通信
    # import socket
    # sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    # command = f"YAW:{yaw_angle:.2f},PITCH:{pitch_angle:.2f}"
    # sock.sendto(command.encode(), ('192.168.1.100', 8080))

def is_target_centered(yaw_angle, pitch_angle, tolerance=1.0):
    """
    检查目标是否已经居中
    
    :param yaw_angle: 当前yaw角度偏差
    :param pitch_angle: 当前pitch角度偏差
    :param tolerance: 容忍角度偏差（度）
    :return: True如果目标已居中
    """
    return abs(yaw_angle) < tolerance and abs(pitch_angle) < tolerance

def calculate_gimbal_speed(angle_error, max_speed=10.0, min_speed=0.5):
    """
    根据角度误差计算云台旋转速度
    
    :param angle_error: 角度误差（度）
    :param max_speed: 最大速度
    :param min_speed: 最小速度
    :return: 计算出的速度
    """
    # 简单的比例控制
    speed = min(max_speed, max(min_speed, abs(angle_error) * 2))
    return speed if angle_error >= 0 else -speed

print("云台控制功能已加载！")

云台控制功能已加载！


### 使用说明和参数调整指南

#### 功能说明：
1. **矩形中心检测**：使用轮廓质心计算精确的矩形中心点
2. **距离估计**：基于矩形在图像中的像素尺寸和已知实际尺寸计算距离
3. **云台角度计算**：根据目标相对于图像中心的偏差计算yaw和pitch角度

#### 重要参数需要根据实际情况调整：

1. **摄像头参数**（在GimbalConfig类中）：
   - `camera_fov_horizontal`和`camera_fov_vertical`：摄像头的水平和垂直视野角度
   - 可以通过摄像头规格书获得，或者实测得出

2. **目标尺寸**：
   - `real_rect_width`和`real_rect_height`：目标矩形的实际物理尺寸（毫米）
   - 必须准确测量你要瞄准的目标尺寸

3. **检测阈值**：
   - `area > 500`：最小面积阈值，过滤噪声
   - `convexity_ratio > 0.92`：凸性比值阈值

#### 输出信息：
- **Distance**：估计的目标距离（毫米）
- **Yaw**：水平方向需要旋转的角度（正值向右，负值向左）
- **Pitch**：垂直方向需要旋转的角度（正值向上，负值向下）

#### 控制台输出：
程序会在控制台实时打印云台角度信息，你可以将这些信息发送给云台控制器。

#### 按键控制：
- 按 'q' 退出程序
- 按 's' 保存当前检测帧（当有目标时）

In [17]:
# 改进的距离检测算法 - 基于平行边长度计算
def calculate_parallel_edges_length(approx_contour):
    """
    计算四边形中两对平行边的长度
    
    :param approx_contour: 近似的四边形轮廓点 (4个点)
    :return: (avg_width, avg_height) 平均宽度和高度（像素）
    """
    if len(approx_contour) != 4:
        return None, None
    
    # 获取四个顶点坐标
    points = approx_contour.reshape(4, 2)
    
    # 计算所有边的长度
    edge_lengths = []
    for i in range(4):
        p1 = points[i]
        p2 = points[(i + 1) % 4]
        length = np.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)
        edge_lengths.append(length)
    
    # 找到对边（平行边）
    # 方法：计算所有边的向量，找到夹角最小的两对边
    vectors = []
    for i in range(4):
        p1 = points[i]
        p2 = points[(i + 1) % 4]
        vector = np.array([p2[0] - p1[0], p2[1] - p1[1]])
        vectors.append(vector)
    
    # 计算向量间的角度差异，找到平行边对
    def angle_between_vectors(v1, v2):
        """计算两个向量之间的角度"""
        cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
        cos_angle = np.clip(cos_angle, -1, 1)  # 防止数值误差
        return np.arccos(np.abs(cos_angle))  # 取绝对值，因为我们关心的是平行性
    
    # 寻找最平行的边对
    min_angle_pairs = []
    for i in range(4):
        for j in range(i + 2, 4):  # 只考虑对边
            if j == i + 2 or (i == 0 and j == 3):  # 确保是对边
                continue
            angle = angle_between_vectors(vectors[i], vectors[j])
            min_angle_pairs.append((i, j, angle, edge_lengths[i], edge_lengths[j]))
    
    # 简化：直接按照四边形的对边来计算
    # 边0和边2为一对，边1和边3为一对
    pair1_avg = (edge_lengths[0] + edge_lengths[2]) / 2  # 第一对对边的平均长度
    pair2_avg = (edge_lengths[1] + edge_lengths[3]) / 2  # 第二对对边的平均长度
    
    # 返回较长和较短的边作为宽度和高度
    if pair1_avg > pair2_avg:
        return pair1_avg, pair2_avg  # width, height
    else:
        return pair2_avg, pair1_avg  # width, height

def calculate_distance_from_parallel_edges(approx_contour, config):
    """
    基于平行边长度计算距离
    
    :param approx_contour: 近似的四边形轮廓点
    :param config: 云台配置对象
    :return: 估计的距离 (毫米)
    """
    avg_width, avg_height = calculate_parallel_edges_length(approx_contour)
    
    if avg_width is None or avg_height is None:
        return None
    
    # 使用相似三角形原理计算距离
    # distance = (real_size * image_size) / (pixel_size * 2 * tan(fov/2))
    
    # 计算基于宽度的距离
    distance_from_width = (config.real_rect_width * config.image_width) / \
                         (avg_width * 2 * np.tan(np.radians(config.camera_fov_horizontal / 2)))
    
    # 计算基于高度的距离
    distance_from_height = (config.real_rect_height * config.image_height) / \
                          (avg_height * 2 * np.tan(np.radians(config.camera_fov_vertical / 2)))
    
    # 取平均值作为最终距离估计
    estimated_distance = (distance_from_width + distance_from_height) / 2
    
    return estimated_distance

def draw_contour_edges_info(frame, approx_contour, rect_center):
    """
    在图像上绘制轮廓边长信息
    
    :param frame: 图像帧
    :param approx_contour: 近似轮廓
    :param rect_center: 矩形中心点
    """
    if len(approx_contour) != 4:
        return
    
    points = approx_contour.reshape(4, 2)
    avg_width, avg_height = calculate_parallel_edges_length(approx_contour)
    
    # if avg_width is not None and avg_height is not None:
    #     # 在矩形中心附近显示边长信息
    #     cv2.putText(frame, f'W:{avg_width:.1f}px', 
    #                (rect_center[0] - 30, rect_center[1] + 15), 
    #                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 255), 1)
    #     cv2.putText(frame, f'H:{avg_height:.1f}px', 
    #                (rect_center[0] - 30, rect_center[1] + 30), 
    #                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 255), 1)
    
    # 绘制每条边的长度
    for i in range(4):
        p1 = points[i]
        p2 = points[(i + 1) % 4]
        edge_length = np.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)
        
        # 在边的中点显示长度
        mid_point = ((p1[0] + p2[0]) // 2, (p1[1] + p2[1]) // 2)
        cv2.putText(frame, f'{edge_length:.0f}', mid_point, 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 255), 1)

print("改进的距离检测算法已加载完成！")

改进的距离检测算法已加载完成！


In [19]:
# 使用改进距离算法的完整检测循环
cap.set(cv2.CAP_PROP_AUTO_EXPOSURE, 0.25)  # Set auto exposure to manual mode
cap.set(cv2.CAP_PROP_EXPOSURE, -8)  # Set exposure value (adjust as needed)
# cap.set(cv2.CAP_PROP_AUTO_EXPOSURE, 1)  # Set auto exposure to  mode

# 用于存储目标信息的变量
target_found = False
target_center = (0, 0)
target_distance = 0
yaw_angle = 0
pitch_angle = 0

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break
    
    frame = cv2.resize(frame, (config.image_width, config.image_height))
    
    # 图像处理
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # img = cv2.equalizeHist(img)
    img = cv2.GaussianBlur(img, (5, 5), 0)
    bimg = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 7)
    
    # 矩形检测
    contours, _ = cv2.findContours(bimg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # 在原图上绘制检测到的矩形
    result_frame = frame.copy()
    # 绘制图像中心十字线
    cv2.line(result_frame, (config.image_center_x - 10, config.image_center_y), 
             (config.image_center_x + 10, config.image_center_y), (255, 255, 255), 2)
    cv2.line(result_frame, (config.image_center_x, config.image_center_y - 10), 
             (config.image_center_x, config.image_center_y + 10), (255, 255, 255), 2)
    
    target_found = False
    best_target = None
    best_score = 0
    
    for contour in contours:
        # 计算轮廓面积，过滤小的噪声
        area = cv2.contourArea(contour)
        if area > 500:  # 可以调整这个阈值
            # 计算轮廓的近似多边形
            epsilon = 0.02 * cv2.arcLength(contour, True)
            approx = cv2.approxPolyDP(contour, epsilon, True)
            
            # 如果近似多边形有4个顶点，认为是矩形
            if len(approx) == 4:
                # 检查是否为凸多边形
                is_convex = cv2.isContourConvex(approx)
                
                if is_convex:  # 只保留凸多边形
                    # 计算凸包面积与轮廓面积的比值来进一步验证凸性
                    hull = cv2.convexHull(contour)
                    hull_area = cv2.contourArea(hull)
                    convexity_ratio = area / hull_area if hull_area > 0 else 0
                    
                    # 凸性比值应该接近1（凸多边形的面积与其凸包面积相等）
                    if convexity_ratio > 0.92:  # 调整阈值以过滤轻微的凹陷
                        # 获取矩形信息
                        rect_center_x, rect_center_y = get_rectangle_center(contour)
                        
                        # 使用改进的距离计算算法
                        estimated_distance = calculate_distance_from_parallel_edges(approx, config)
                        
                        if estimated_distance is not None:  # 确保距离计算成功
                            # 计算云台角度
                            yaw, pitch = calculate_gimbal_angles(rect_center_x, rect_center_y, estimated_distance, config)
                            
                            # 计算平行边长度用于显示
                            avg_width, avg_height = calculate_parallel_edges_length(approx)
                            
                            # 计算目标评分（距离图像中心越近分数越高，面积越大分数越高）
                            center_distance = np.sqrt((rect_center_x - config.image_center_x)**2 + 
                                                     (rect_center_y - config.image_center_y)**2)
                            score = area / (center_distance + 1)  # 面积除以到中心的距离
                            
                            # 选择最佳目标
                            if score > best_score:
                                best_score = score
                                best_target = {
                                    'contour': contour,
                                    'approx': approx,
                                    'center': (rect_center_x, rect_center_y),
                                    'area': area,
                                    'distance': estimated_distance,
                                    'yaw': yaw,
                                    'pitch': pitch,
                                    'convexity': convexity_ratio,
                                    'avg_width': avg_width,
                                    'avg_height': avg_height
                                }
                                target_found = True
                            
                            # 绘制所有检测到的矩形（绿色）
                            # cv2.drawContours(result_frame, [approx], -1, (0, 255, 0), 1)
                             
                            # 绘制边长信息
                            draw_contour_edges_info(result_frame, approx, (rect_center_x, rect_center_y))
                            
                            # 显示距离和角度信息
                            x, y, w, h = cv2.boundingRect(contour)
                            cv2.putText(result_frame, f'D:{int(estimated_distance)}mm', (x, y-55), 
                                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)
                            cv2.putText(result_frame, f'Y:{yaw:.1f} P:{pitch:.1f}', (x, y-40), 
                                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)
                            cv2.putText(result_frame, f'Area:{int(area)}', (x, y-25), 
                                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)
                            # cv2.putText(result_frame, f'Convex:{convexity_ratio:.2f}', (x, y-10), 
                            #            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
    
    # 如果找到最佳目标，用红色突出显示
    if target_found and best_target:
        # 更新全局目标信息
        target_center = best_target['center']
        target_distance = best_target['distance']
        yaw_angle = best_target['yaw']
        pitch_angle = best_target['pitch']
        
        # 用红色突出显示最佳目标
        cv2.drawContours(result_frame, [best_target['approx']], -1, (0, 0, 255), 1)
        cv2.circle(result_frame, target_center, 1, (0, 0, 255), -1)
        
        # 绘制从图像中心到目标中心的连线
        cv2.line(result_frame, (config.image_center_x, config.image_center_y), 
                 target_center, (0, 0, 255), 2)
        
        # 显示最佳目标的详细信息
        # cv2.putText(result_frame, f'TGT', (target_center[0]-50, target_center[1]-70), 
        #            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
        
        # 在屏幕顶部显示云台控制信息
        cv2.putText(result_frame, f'Distance: {int(target_distance)}mm', (10, 20), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        cv2.putText(result_frame, f'Yaw: {yaw_angle:.2f} deg', (10, 40), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        cv2.putText(result_frame, f'Pitch: {pitch_angle:.2f} deg', (10, 60), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        # 显示平行边长度信息
        if best_target['avg_width'] and best_target['avg_height']:
            cv2.putText(result_frame, f'AvgW: {best_target["avg_width"]:.1f}px', (10, 80), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 255), 1)
            cv2.putText(result_frame, f'AvgH: {best_target["avg_height"]:.1f}px', (10, 95), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 255), 1)
    else:
        # 没有找到目标时的显示
        cv2.putText(result_frame, 'NO TARGET FOUND', (10, 30), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
    
    # 显示图像
    cv2.imshow("Binary Image", bimg)
    cv2.imshow("Improved Gimbal Detection", result_frame)
    
    # 在控制台输出云台角度信息（可用于发送给云台控制器）
    if target_found and best_target:
        print(f"Target found - Yaw: {yaw_angle:.2f}°, Pitch: {pitch_angle:.2f}°, Distance: {int(target_distance)}mm, "
              f"AvgW: {best_target['avg_width']:.1f}px, AvgH: {best_target['avg_height']:.1f}px")
    
    # 按'q'退出，按's'保存当前帧，按'd'切换调试模式
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        cv2.destroyAllWindows()
        break
    elif key == ord('s') and target_found:
        cv2.imwrite(f'improved_target_{int(target_distance)}mm.jpg', result_frame)
        print(f"Frame saved with improved detection at {int(target_distance)}mm")

Target found - Yaw: -23.91°, Pitch: 6.67°, Distance: 1590mm, AvgW: 33.1px, AvgH: 26.0px
Target found - Yaw: -24.19°, Pitch: 6.67°, Distance: 1535mm, AvgW: 34.2px, AvgH: 27.0px
Target found - Yaw: -24.19°, Pitch: 6.67°, Distance: 1506mm, AvgW: 35.6px, AvgH: 27.0px
Target found - Yaw: -24.19°, Pitch: 6.67°, Distance: 1457mm, AvgW: 36.7px, AvgH: 28.0px
Target found - Yaw: -24.75°, Pitch: 6.93°, Distance: 1409mm, AvgW: 37.7px, AvgH: 29.2px
Target found - Yaw: -25.03°, Pitch: 6.93°, Distance: 1390mm, AvgW: 38.3px, AvgH: 29.5px
Target found - Yaw: -25.31°, Pitch: 6.67°, Distance: 1370mm, AvgW: 38.7px, AvgH: 30.0px
Target found - Yaw: -25.59°, Pitch: 6.67°, Distance: 1359mm, AvgW: 38.7px, AvgH: 30.5px
Target found - Yaw: -25.59°, Pitch: 6.93°, Distance: 1351mm, AvgW: 39.2px, AvgH: 30.5px
Target found - Yaw: -25.59°, Pitch: 7.20°, Distance: 1333mm, AvgW: 40.3px, AvgH: 30.5px
Target found - Yaw: -25.88°, Pitch: 7.73°, Distance: 1328mm, AvgW: 40.7px, AvgH: 30.5px
Target found - Yaw: -25.88°, Pit

In [44]:
cap.release()

In [63]:

cv2.destroyAllWindows()

### 第四版：加入 OpenCV 卡尔曼滤波并绘制预测坐标
- 使用 `cv2.KalmanFilter(4, 2)`，状态为 `(x, y, vx, vy)`
- 每帧先 `predict()` 得到预测坐标，再在检测到目标时 `correct()` 更新
- 图像上会显示：
  - 红点：当前测量中心（Measurement）
  - 蓝点：卡尔曼预测点（Prediction）
  - 黄点：校正后的估计点（Corrected）

In [26]:
# 第四版：卡尔曼滤波预测目标中心并绘制预测坐标
cap.set(cv2.CAP_PROP_AUTO_EXPOSURE, 0.25)
cap.set(cv2.CAP_PROP_EXPOSURE, -8)

# 1) 初始化 OpenCV 卡尔曼滤波器
kalman = cv2.KalmanFilter(4, 2)  # state: x,y,vx,vy ; measurement: x,y
kalman.transitionMatrix = np.array([[1, 0, 30, 0],
                                    [0, 1, 0, 30],
                                    [0, 0, 1, 0],
                                    [0, 0, 0, 1]], dtype=np.float32)
kalman.measurementMatrix = np.array([[1, 0, 0, 0],
                                     [0, 1, 0, 0]], dtype=np.float32)
kalman.processNoiseCov = np.eye(4, dtype=np.float32) * 1e-2
kalman.measurementNoiseCov = np.eye(2, dtype=np.float32) * 4e0
kalman.errorCovPost = np.eye(4, dtype=np.float32) * 1.0
kalman.statePost = np.array([[config.image_center_x],
                             [config.image_center_y],
                             [0],
                             [0]], dtype=np.float32)

target_distance = 1000.0  # 当帧未测得距离时保留上次值
print("Kalman tracking started. Press 'q' to quit, 's' to save frame.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break

    frame = cv2.resize(frame, (config.image_width, config.image_height))
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    img = cv2.GaussianBlur(img, (5, 5), 0)
    bimg = cv2.adaptiveThreshold(
        img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 7
    )

    contours, _ = cv2.findContours(bimg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    result_frame = frame.copy()

    # 画图像中心十字
    cv2.line(
        result_frame,
        (config.image_center_x - 10, config.image_center_y),
        (config.image_center_x + 10, config.image_center_y),
        (255, 255, 255),
        1,
    )
    cv2.line(
        result_frame,
        (config.image_center_x, config.image_center_y - 10),
        (config.image_center_x, config.image_center_y + 10),
        (255, 255, 255),
        1,
    )

    # 2) 先预测
    prediction = kalman.predict()
    pred_x = int(np.clip(prediction[0, 0], 0, config.image_width - 1))
    pred_y = int(np.clip(prediction[1, 0], 0, config.image_height - 1))

    # 当前帧找最佳目标
    best_target = None
    best_score = 0.0

    for contour in contours:
        area = cv2.contourArea(contour)
        if area <= 500:
            continue

        epsilon = 0.02 * cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, epsilon, True)
        if len(approx) != 4:
            continue

        if not cv2.isContourConvex(approx):
            continue

        hull = cv2.convexHull(contour)
        hull_area = cv2.contourArea(hull)
        convexity_ratio = area / hull_area if hull_area > 0 else 0
        if convexity_ratio <= 0.92:
            continue

        rect_center_x, rect_center_y = get_rectangle_center(contour)
        estimated_distance = calculate_distance_from_parallel_edges(approx, config)
        if estimated_distance is None:
            continue

        yaw, pitch = calculate_gimbal_angles(rect_center_x, rect_center_y, estimated_distance, config)
        center_distance = np.sqrt(
            (rect_center_x - config.image_center_x) ** 2
            + (rect_center_y - config.image_center_y) ** 2
        )
        score = area / (center_distance + 1.0)

        if score > best_score:
            best_score = score
            best_target = {
                "contour": contour,
                "approx": approx,
                "center": (rect_center_x, rect_center_y),
                "distance": estimated_distance,
                "yaw": yaw,
                "pitch": pitch,
            }

    # 3) 如果检测到目标，进行校正
    if best_target is not None:
        meas_x, meas_y = best_target["center"]
        target_distance = best_target["distance"]

        measurement = np.array([[np.float32(meas_x)], [np.float32(meas_y)]], dtype=np.float32)
        corrected = kalman.correct(measurement)
        corr_x = int(np.clip(corrected[0, 0], 0, config.image_width - 1))
        corr_y = int(np.clip(corrected[1, 0], 0, config.image_height - 1))

        # 绘制目标轮廓
        cv2.drawContours(result_frame, [best_target["approx"]], -1, (0, 0, 255), 1)

        # 红点：测量值
        cv2.circle(result_frame, (meas_x, meas_y), 3, (0, 0, 255), -1)
        cv2.putText(result_frame, "Meas", (meas_x + 6, meas_y - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

        # 黄点：校正后估计值
        cv2.circle(result_frame, (corr_x, corr_y), 4, (0, 255, 255), -1)
        cv2.putText(result_frame, "Corr", (corr_x + 6, corr_y + 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)

        # 蓝点：预测值
        cv2.circle(result_frame, (pred_x, pred_y), 4, (255, 0, 0), -1)
        cv2.putText(result_frame, "Pred", (pred_x + 6, pred_y - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)

        # 显示角度和距离（使用校正点计算角度更稳定）
        yaw_kf, pitch_kf = calculate_gimbal_angles(corr_x, corr_y, target_distance, config)
        cv2.putText(result_frame, f"Distance: {int(target_distance)}mm", (10, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        cv2.putText(result_frame, f"Yaw(KF): {yaw_kf:.2f} deg", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        cv2.putText(result_frame, f"Pitch(KF): {pitch_kf:.2f} deg", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        print(f"KF Target - Pred({pred_x},{pred_y}) Corr({corr_x},{corr_y}) "
              f"Yaw:{yaw_kf:.2f} Pitch:{pitch_kf:.2f} Dist:{int(target_distance)}mm")

    else:
        # 未检测到目标也继续显示预测点（便于观察滤波器预测轨迹）
        cv2.circle(result_frame, (pred_x, pred_y), 4, (255, 0, 0), -1)
        cv2.putText(result_frame, "Pred only", (pred_x + 6, pred_y - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)
        cv2.putText(result_frame, "NO TARGET FOUND", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    cv2.imshow("Binary Image", bimg)
    cv2.imshow("Kalman Gimbal Detection", result_frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        cv2.destroyAllWindows()
        break
    elif key == ord('s'):
        cv2.imwrite(f'kalman_target_{int(target_distance)}mm.jpg', result_frame)
        print(f"Frame saved at {int(target_distance)}mm")

Kalman tracking started. Press 'q' to quit, 's' to save frame.
KF Target - Pred(160,120) Corr(109,185) Yaw:-14.34 Pitch:-17.33 Dist:2018mm
KF Target - Pred(107,187) Corr(99,177) Yaw:-17.16 Pitch:-15.20 Dist:1884mm
KF Target - Pred(89,169) Corr(89,169) Yaw:-19.97 Pitch:-13.07 Dist:1773mm
KF Target - Pred(80,162) Corr(81,162) Yaw:-22.22 Pitch:-11.20 Dist:1695mm
KF Target - Pred(73,155) Corr(75,155) Yaw:-23.91 Pitch:-9.33 Dist:1639mm
KF Target - Pred(68,148) Corr(70,148) Yaw:-25.31 Pitch:-7.47 Dist:1629mm
KF Target - Pred(65,140) Corr(67,139) Yaw:-26.16 Pitch:-5.07 Dist:1572mm
KF Target - Pred(63,130) Corr(64,130) Yaw:-27.00 Pitch:-2.67 Dist:1515mm
KF Target - Pred(61,122) Corr(62,123) Yaw:-27.56 Pitch:-0.80 Dist:1481mm
KF Target - Pred(60,116) Corr(63,116) Yaw:-27.28 Pitch:1.07 Dist:1420mm
KF Target - Pred(63,109) Corr(67,110) Yaw:-26.16 Pitch:2.67 Dist:1398mm
KF Target - Pred(69,104) Corr(72,105) Yaw:-24.75 Pitch:4.00 Dist:1351mm
KF Target - Pred(77,100) Corr(79,102) Yaw:-22.78 Pitch:4.

### 第五版：基于加速度的超前预测
- 使用 `cv2.KalmanFilter(6, 2)`，状态为 `(x, y, vx, vy, ax, ay)`
- 检测值仍然只使用矩形中心 `(x, y)`
- 先用卡尔曼滤波估计位置、速度、加速度，再按设定超前时间计算未来坐标
- 图像上会显示：
  - 红点：当前测量中心
  - 黄点：当前滤波估计中心
  - 青点：基于加速度模型的超前预测点

In [ ]:
# 第五版：基于加速度的超前预测
import time

cap.set(cv2.CAP_PROP_AUTO_EXPOSURE, 0.25)
cap.set(cv2.CAP_PROP_EXPOSURE, -8)

# 常加速度模型: state = [x, y, vx, vy, ax, ay]^T
kf_acc = cv2.KalmanFilter(6, 2)

def update_acc_transition_matrix(kalman_filter, dt):
    kalman_filter.transitionMatrix = np.array([
        [1, 0, dt, 0, 0.5 * dt * dt, 0],
        [0, 1, 0, dt, 0, 0.5 * dt * dt],
        [0, 0, 1, 0, dt, 0],
        [0, 0, 0, 1, 0, dt],
        [0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 1],
    ], dtype=np.float32)

def build_lookahead_transition(dt):
    return np.array([
        [1, 0, dt, 0, 0.5 * dt * dt, 0],
        [0, 1, 0, dt, 0, 0.5 * dt * dt],
        [0, 0, 1, 0, dt, 0],
        [0, 0, 0, 1, 0, dt],
        [0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 1],
    ], dtype=np.float32)

kf_acc.measurementMatrix = np.array([
    [1, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0],
], dtype=np.float32)
kf_acc.processNoiseCov = np.diag([1e-2, 1e-2, 8e-2, 8e-2, 2e-1, 2e-1]).astype(np.float32)
kf_acc.measurementNoiseCov = np.eye(2, dtype=np.float32) * 4.0
kf_acc.errorCovPost = np.eye(6, dtype=np.float32) * 1.0
kf_acc.statePost = np.array([
    [config.image_center_x],
    [config.image_center_y],
    [0],
    [0],
    [0],
    [0],
], dtype=np.float32)

lead_time = 0.15  # 单位: 秒, 越大预测越超前
target_distance = 1000.0
last_time = time.perf_counter()

print("Acceleration lead prediction started. Press 'q' to quit, 's' to save frame.")

while True:
    current_time = time.perf_counter()
    dt = current_time - last_time
    last_time = current_time
    dt = float(np.clip(dt, 1.0 / 120.0, 0.1))

    update_acc_transition_matrix(kf_acc, dt)

    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break

    frame = cv2.resize(frame, (config.image_width, config.image_height))
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    img = cv2.GaussianBlur(img, (5, 5), 0)
    bimg = cv2.adaptiveThreshold(
        img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 7
    )

    contours, _ = cv2.findContours(bimg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    result_frame = frame.copy()

    cv2.line(
        result_frame,
        (config.image_center_x - 10, config.image_center_y),
        (config.image_center_x + 10, config.image_center_y),
        (255, 255, 255),
        1,
    )
    cv2.line(
        result_frame,
        (config.image_center_x, config.image_center_y - 10),
        (config.image_center_x, config.image_center_y + 10),
        (255, 255, 255),
        1,
    )

    predicted_state = kf_acc.predict()
    pred_x = int(np.clip(predicted_state[0, 0], 0, config.image_width - 1))
    pred_y = int(np.clip(predicted_state[1, 0], 0, config.image_height - 1))

    best_target = None
    best_score = 0.0

    for contour in contours:
        area = cv2.contourArea(contour)
        if area <= 500:
            continue

        epsilon = 0.02 * cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, epsilon, True)
        if len(approx) != 4:
            continue

        if not cv2.isContourConvex(approx):
            continue

        hull = cv2.convexHull(contour)
        hull_area = cv2.contourArea(hull)
        convexity_ratio = area / hull_area if hull_area > 0 else 0
        if convexity_ratio <= 0.92:
            continue

        rect_center_x, rect_center_y = get_rectangle_center(contour)
        estimated_distance = calculate_distance_from_parallel_edges(approx, config)
        if estimated_distance is None:
            continue

        yaw, pitch = calculate_gimbal_angles(rect_center_x, rect_center_y, estimated_distance, config)
        center_distance = np.sqrt(
            (rect_center_x - config.image_center_x) ** 2
            + (rect_center_y - config.image_center_y) ** 2
        )
        score = area / (center_distance + 1.0)

        if score > best_score:
            best_score = score
            best_target = {
                "contour": contour,
                "approx": approx,
                "center": (rect_center_x, rect_center_y),
                "distance": estimated_distance,
                "yaw": yaw,
                "pitch": pitch,
            }

    if best_target is not None:
        meas_x, meas_y = best_target["center"]
        target_distance = best_target["distance"]

        measurement = np.array([[np.float32(meas_x)], [np.float32(meas_y)]], dtype=np.float32)
        corrected_state = kf_acc.correct(measurement)

        corr_x = int(np.clip(corrected_state[0, 0], 0, config.image_width - 1))
        corr_y = int(np.clip(corrected_state[1, 0], 0, config.image_height - 1))

        # 从当前估计状态再向前外推 lead_time 秒
        lead_transition = build_lookahead_transition(lead_time)
        lead_state = lead_transition @ corrected_state
        lead_x = int(np.clip(lead_state[0, 0], 0, config.image_width - 1))
        lead_y = int(np.clip(lead_state[1, 0], 0, config.image_height - 1))

        # 当前估计的速度和加速度，便于调试
        vel_x = float(corrected_state[2, 0])
        vel_y = float(corrected_state[3, 0])
        acc_x = float(corrected_state[4, 0])
        acc_y = float(corrected_state[5, 0])

        cv2.drawContours(result_frame, [best_target["approx"]], -1, (0, 0, 255), 1)

        # 红点：测量点
        cv2.circle(result_frame, (meas_x, meas_y), 3, (0, 0, 255), -1)
        cv2.putText(result_frame, "Meas", (meas_x + 6, meas_y - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

        # 黄点：滤波校正点
        cv2.circle(result_frame, (corr_x, corr_y), 4, (0, 255, 255), -1)
        cv2.putText(result_frame, "KF", (corr_x + 6, corr_y + 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)

        # 青点：基于加速度的超前预测点
        cv2.circle(result_frame, (lead_x, lead_y), 5, (255, 255, 0), -1)
        cv2.putText(result_frame, "Lead", (lead_x + 6, lead_y - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 0), 1)

        # 蓝点：当前时刻预测点
        cv2.circle(result_frame, (pred_x, pred_y), 3, (255, 0, 0), -1)
        cv2.putText(result_frame, "Pred", (pred_x + 6, pred_y + 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 0, 0), 1)

        cv2.line(result_frame, (corr_x, corr_y), (lead_x, lead_y), (255, 255, 0), 1)

        yaw_lead, pitch_lead = calculate_gimbal_angles(lead_x, lead_y, target_distance, config)

        cv2.putText(result_frame, f"Distance: {int(target_distance)}mm", (10, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        cv2.putText(result_frame, f"Yaw(lead): {yaw_lead:.2f} deg", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        cv2.putText(result_frame, f"Pitch(lead): {pitch_lead:.2f} deg", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        cv2.putText(result_frame, f"Vx:{vel_x:.1f} Vy:{vel_y:.1f}", (10, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 1)
        cv2.putText(result_frame, f"Ax:{acc_x:.1f} Ay:{acc_y:.1f}", (10, 98),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 0), 1)

        print(
            f"Lead Target - Meas({meas_x},{meas_y}) KF({corr_x},{corr_y}) Lead({lead_x},{lead_y}) "
            f"Yaw:{yaw_lead:.2f} Pitch:{pitch_lead:.2f} Dist:{int(target_distance)}mm "
            f"V=({vel_x:.2f},{vel_y:.2f}) A=({acc_x:.2f},{acc_y:.2f})"
        )

    else:
        # 丢目标时保留当前预测，继续做短时前向外推
        lead_transition = build_lookahead_transition(lead_time)
        lead_state = lead_transition @ predicted_state
        lead_x = int(np.clip(lead_state[0, 0], 0, config.image_width - 1))
        lead_y = int(np.clip(lead_state[1, 0], 0, config.image_height - 1))

        cv2.circle(result_frame, (pred_x, pred_y), 3, (255, 0, 0), -1)
        cv2.putText(result_frame, "Pred", (pred_x + 6, pred_y + 12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 0, 0), 1)
        cv2.circle(result_frame, (lead_x, lead_y), 5, (255, 255, 0), -1)
        cv2.putText(result_frame, "Lead only", (lead_x + 6, lead_y - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)
        cv2.putText(result_frame, "NO TARGET FOUND", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    cv2.imshow("Binary Image", bimg)
    cv2.imshow("Acceleration Lead Prediction", result_frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        cv2.destroyAllWindows()
        break
    elif key == ord('s'):
        cv2.imwrite(f'acc_lead_target_{int(target_distance)}mm.jpg', result_frame)
        print(f"Frame saved at {int(target_distance)}mm")